# Task 2



### Brightspace code for Youtube to MP3 extraction

In [5]:
# importing packages
from pytubefix import YouTube
import os

# url input from youtube
yt = YouTube("https://www.youtube.com/watch?v=ozj4T5M5GTk")

# extract only audio
video = yt.streams.filter(only_audio=True).first()

# set destination to save file
destination = ("MP3_Video")

# download the file
out_file = video.download(output_path=destination)

# save the file
base, ext = os.path.splitext(out_file)
new_file = 'recording.mp3'
os.rename(out_file, new_file)

### Whisper transcription tool test

Required to install:

pip install -U openai-whisper

and for conda environment

conda install -c conda-forge ffmpeg


In [9]:
import whisper

model = whisper.load_model("base")
result = model.transcribe('recording.mp3')
print(result["text"])

 LeBurn, California. Just 30 miles from Los Angeles, this suburb is home to a neighborhood Italian bistro called Charles. And in 2010, it was bought by the Lava family. Would you like a booth? I started working here 10 years ago. I was a waitress here. And then I had the opportunity to take it over. But I didn't have the money or the collateral. So I had to go to my mom and my sister. Pat and Val, both full-time schoolteachers, financed the restaurant and put their houses up as collateral to help Tatiana to fill her dream. My mom and I don't know about the restaurant business. And so we put a lot of trust in my sister's experience to run the restaurant. Daddy, are we going to have enough bread for tonight? Of course. When I took over the restaurant, I didn't want to change the menu or the chef. I just wanted to keep it as is because I love this restaurant. How are we doing? It's not good. But it didn't pan out the way I wanted it to. I mean, I could sit here all day and have maybe four

In [11]:
result['text']

" LeBurn, California. Just 30 miles from Los Angeles, this suburb is home to a neighborhood Italian bistro called Charles. And in 2010, it was bought by the Lava family. Would you like a booth? I started working here 10 years ago. I was a waitress here. And then I had the opportunity to take it over. But I didn't have the money or the collateral. So I had to go to my mom and my sister. Pat and Val, both full-time schoolteachers, financed the restaurant and put their houses up as collateral to help Tatiana to fill her dream. My mom and I don't know about the restaurant business. And so we put a lot of trust in my sister's experience to run the restaurant. Daddy, are we going to have enough bread for tonight? Of course. When I took over the restaurant, I didn't want to change the menu or the chef. I just wanted to keep it as is because I love this restaurant. How are we doing? It's not good. But it didn't pan out the way I wanted it to. I mean, I could sit here all day and have maybe fou

In [1]:
import re
import csv
import whisper
from pathlib import Path

def split_into_sentences(text):
    """Split text into sentences using simple regex"""
    # Split on sentence-ending punctuation followed by space and capital letter
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    
    # Clean and filter sentences
    cleaned_sentences = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) > 5:  # Filter very short sentences
            cleaned_sentences.append(sentence)
    
    return cleaned_sentences

def save_to_csv(sentences, output_path):
    """Save sentences to CSV file"""
    with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Sentence'])  # Header
        
        for sentence in sentences:
            writer.writerow([sentence])

def main(mp3_file):
    """
    Main function: convert MP3 to sentences CSV
    
    Args:
        mp3_file (str): Path to MP3 file
    
    Returns:
        str: Path to saved CSV file
    """
    # Load Whisper model and transcribe
    model = whisper.load_model("medium.en")
    result = model.transcribe(mp3_file, fp16=False)
    
    # Split transcription into sentences
    sentences = split_into_sentences(result["text"])
    
    # Create output filename
    input_path = Path(mp3_file)
    output_path = input_path.parent / f"{input_path.stem}_sentences.csv"
    
    # Save to CSV
    save_to_csv(sentences, output_path)
    
    print(f"Transcribed {len(sentences)} sentences to {output_path}")
    return str(output_path)

In [2]:
main('recording.mp3')

c:\Users\filip\anaconda3\envs\Y2_BlockA\lib\site-packages\whisper\__init__.py:69: UserWarning: C:\Users\filip\.cache\whisper\medium.en.pt exists, but the SHA256 checksum does not match; re-downloading the file
  warnings.warn(
  0%|                                    | 440k/1.42G [00:08<7:39:48, 55.4kiB/s]


KeyboardInterrupt: 

### AssemlyAI transcription tool test

In [ ]:
# Install the requests package by executing the command "pip install requests"
import requests
import time

base_url = "https://api.assemblyai.com"

headers = {
    "authorization": "REDACTED"
}
# You can upload a local file using the following code
with open("Data\Recordings\recording.mp3", "rb") as f:
   response = requests.post(base_url + "/v2/upload",
                           headers=headers,
                           data=f)
 
audio_url = response.json()["upload_url"]

data = {
    "audio_url": audio_url,
    "speech_model": "universal"
}

url = base_url + "/v2/transcript"
response = requests.post(url, json=data, headers=headers)

transcript_id = response.json()['id']
polling_endpoint = base_url + "/v2/transcript/" + transcript_id

while True:
  transcription_result = requests.get(polling_endpoint, headers=headers).json()
  transcript_text = transcription_result['text']

  if transcription_result['status'] == 'completed':
    print(f"Transcript Text:", transcript_text)
    break

  elif transcription_result['status'] == 'error':
    raise RuntimeError(f"Transcription failed: {transcription_result['error']}")

  else:
    time.sleep(3)